In [ ]:
from datetime import datetime

import pyspark.sql.functions as F
import lightgbm as lgb
import mlflow
import mlflow.lightgbm

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

for chave, valor in {
    "data_inicio": "20260501",
    "data_fim": "20260601",
    "catalogo": "workspace",
    "esquema": "default"
}.items():
    dbutils.widgets.text(chave, valor)

inicio = dbutils.widgets.get("data_inicio")
fim = dbutils.widgets.get("data_fim")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

datetime.strptime(inicio, "%Y%m%d")
datetime.strptime(fim, "%Y%m%d")

if inicio >= fim:
    raise ValueError("Periodo invalido.")

tabela = f"{catalogo}.{esquema}.base_tabela_prod_aprimorada"
nome_modelo = f"{catalogo}.{esquema}.propensao_compra_modelo_aprimorado"
target = "comprou_eletronico"

print(f"Re-treino para o periodo {inicio} a {fim}.")

particao = F.col("dia_prtc").cast("string")

df = spark.table(tabela).filter(
    (particao >= inicio) & (particao < fim)
)

pdf = df.toPandas()

if len(pdf) < 30:
    raise ValueError("Registros insuficientes.")

features = [
    coluna
    for coluna in pdf.columns
    if coluna not in ["dia_prtc", target]
]

pdf = pdf[features + [target]].dropna()

X = pdf[features]
y = pdf[target].astype(int)

if len(y) < 30 or y.nunique() != 2:
    raise ValueError("A base precisa conter as duas classes.")

if y.value_counts().min() < 4:
    raise ValueError("Poucos exemplos de uma das classes.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

modelo = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    objective="binary",
    verbosity=-1
)

modelo.fit(X_train, y_train)

probs = modelo.predict_proba(X_test)[:, 1]
auc = float(roc_auc_score(y_test, probs))

print(f"AUC de validacao: {auc:.6f}")

if auc < 0.55:
    raise RuntimeError(
        "Modelo reprovado: AUC abaixo de 0.55. "
        "O Champion anterior sera preservado."
    )

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(
    run_name="retreino_correcao_drift_lgbm"
):
    mlflow.log_param("data_inicio", inicio)
    mlflow.log_param("data_fim", fim)
    mlflow.log_param("tabela_origem", tabela)
    mlflow.log_metric("val_roc_auc", auc)

    assinatura = infer_signature(
        X_train,
        modelo.predict_proba(X_train)[:, 1]
    )

    info = mlflow.lightgbm.log_model(
        lgb_model=modelo,
        name="modelo",
        signature=assinatura,
        input_example=X_train.head(3),
        registered_model_name=nome_modelo
    )

versao = getattr(
    info, "registered_model_version", None
)

if not versao:
    raise RuntimeError(
        "Modelo registrado sem versao confirmada. "
        "Champion nao alterado."
    )

client = MlflowClient()

client.set_registered_model_alias(
    name=nome_modelo,
    alias="Champion",
    version=str(versao)
)

print("Modelo registrado:", nome_modelo)
print("Nova versao Champion:", versao)
print("AUC:", auc)